<a href="https://colab.research.google.com/github/eisbetterthanpi/vision/blob/main/semantic_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://github.com/zhixuanli/segmentation-paper-reading-notes/blob/master/others/Summary%20of%20the%20semantic%20segmentation%20datasets.md
# https://docs.pytorch.org/vision/0.12/datasets.html#image-detection-or-segmentation




In [ ]:
# @title OxfordIIITPet
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

root = "./data"
# transform_img = transforms.Compose([transforms.ToTensor()])
# transform_mask = transforms.Compose([transforms.PILToTensor()])

hw = (256, 256)
# hw = (32, 32)
transform_img = transforms.Compose([transforms.Resize(hw), transforms.ToTensor()])
# transform_mask = transforms.Compose([transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST), transforms.PILToTensor()])
r=8
transform_mask = transforms.Compose([transforms.Resize(tuple(x//r for x in hw), interpolation=transforms.InterpolationMode.NEAREST), transforms.PILToTensor(), lambda x:x-1])

# https://docs.pytorch.org/vision/main/generated/torchvision.datasets.OxfordIIITPet.html
train_data = datasets.OxfordIIITPet(root=root, split="trainval", target_types="segmentation", download=True, transform=transform_img, target_transform=transform_mask)
test_data = datasets.OxfordIIITPet(root=root, split="test", target_types="segmentation", download=True, transform=transform_img, target_transform=transform_mask)
batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

# for i, (imgs, msks) in enumerate(train_data): # bchw, b1hw
#     print(i, imgs.shape, msks.shape)
#     print(i, imgs, msks)
#     break


In [ ]:
# @title sincos_2d
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class sinusodial(nn.Module): # Rotary Positional Embeddings, flexible pos
    def __init__(self, dim, top=torch.pi, base=1000):
        super().__init__()
        self.theta = top / (base ** (torch.arange(0, dim, step=2, device=device) / dim))
        # self.theta = top / (base ** torch.linspace(0, 1, dim//2, device=device))

    def forward(self, pos): # [batch] in [0,1]
        angles = (pos.unsqueeze(-1) * self.theta).unsqueeze(-1) # [seq_len, 1] * [dim // 2] -> [seq_len, dim // 2, 1]
        rot_emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1) # [seq_len, dim // 2, 2]
        return rot_emb.flatten(-2) # [seq_len, dim]


def sinusodial(dim, seq_len=512, top=torch.pi, base=1000):
    theta = top / (base ** (torch.arange(0, dim, step=2) / dim))
    pos = torch.arange(seq_len).unsqueeze(-1)
    angles = (pos * theta)[None,...,None] # [seq_len, 1] * [dim//2] -> [1, seq_len, dim//2, 1]
    rot_emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1).flatten(-2).to(device) # [1, seq_len, dim//2, 2] -> [1, seq_len, dim]
    return rot_emb

# sincos for +
# rope for @qk

# https://github.com/facebookresearch/ijepa/blob/main/src/models/vision_transformer.py
def sincos_2d(embed_dim, hw):
    h,w = hw
    grid = torch.stack(torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device)), axis=0)
    # print(grid[:1].shape, grid[1:].shape)
    emb_h = sincos_1d(embed_dim//2, grid[:1]) # (H*W, D/2)
    emb_w = sincos_1d(embed_dim//2, grid[1:]) # (H*W, D/2)
    pos_embed = torch.cat([emb_h, emb_w], axis=1)  # (H*W, D)
    return pos_embed

def sincos_1d(embed_dim, grid, base=10000):
    # omega = 1/base**(torch.arange(embed_dim//2)*2/embed_dim)
    omega = 1/base**(torch.linspace(0,1, embed_dim//2, device=device))
    # print(omega)
    out = torch.einsum('m,d->md', grid.flatten(), omega)   # (M, D/2), outer product
    pos_embed = torch.cat([torch.sin(out), torch.cos(out)], axis=1)  # (M, D)
    return pos_embed # [M, D]

# # emb = sincos_2d(64, 6)
# emb = sincos_2d(64, (4,6))
# print(emb.shape)
# print(emb)



In [ ]:
# @title attn
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def zero_module(module):
    for p in module.parameters():
        p.detach().zero_()
    return module

class SelfAttn(nn.Module):
    def __init__(self, in_dim, n_heads, d_head=None):
        super().__init__()
        self.n_heads = n_heads
        d_head = d_head or in_dim//n_heads
        d_model = n_heads*d_head
        # self.rope = RoPE1(d_head, seq_len=64, top=torch.pi, base=10000)
        # self.rope = RoPE(d_head, seq_len=64, top=torch.pi, base=10000)
        # self.rope = GoldenGateRoPE2d((8,8), n_heads, d_head, min_freq=.8, max_freq=10, n_zero_freqs=0)
        self.qkv = nn.Linear(in_dim, d_model*3, bias=False)
        self.lin = zero_module(nn.Linear(d_model, in_dim))
        self.scale = d_head**-.5

    def forward(self, x): # [b,t,d]
        # print('SelfAttn fwd', x.shape)
        q,k,v = self.qkv(x).unflatten(-1, (self.n_heads,-1)).transpose(-3,-2).chunk(3, dim=-1) # [b,t,h,3d]->[b,h,t,3d]->[b,h,t,d]
        # q, k = self.rope(q), self.rope(k)

        q, k = q.softmax(dim=-1)*self.scale, k.softmax(dim=-2)
        # q, k = F.sigmoid(q-math.log(t))*self.scale, F.sigmoid(k-math.log(t)) # linear sigmoid attention?
        context = k.transpose(-2,-1) @ v # [batch, n_heads, d_head, d_head]
        x = q @ context # [batch, n_heads, T/num_tok, d_head]

        x = x.transpose(1,2).flatten(2)
        return self.lin(x)


class AttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, drop=0):
        super().__init__()
        self.d_model = d_model
        self.norm1, self.norm2 = nn.RMSNorm(d_model), nn.RMSNorm(d_model) # LayerNorm RMSNorm
        self.drop = nn.Dropout(drop)
        self.attn = SelfAttn(d_model, n_heads) # 16448

        # self.attn = SelfAttn(d_model, n_heads, 8) # 16448
        # self.attn = GLAblock(hidden_size=d_model, expand_k=1, expand_v=1, num_heads=n_heads)
        # self.self = Pooling()
        act = nn.GELU() # ReLU GELU
        # self.ff = nn.Sequential(
        #     *[nn.BatchNorm2d(d_model), act, SeparableConv2d(d_model, d_model),]*3
        #     )
        # self.ff = ResBlock(d_model, kernel=1) # 74112
        # self.ff = UIB(d_model, mult=4) # uib m4 36992, m2 18944
        # self.ff = GLU(d_model, int(3.5*d_model)) # 3.5
        # self.ff = GLU(d_model, d_model) # 3.5
        ff_dim=d_model*4#mult
        self.ff = nn.Sequential(
            nn.RMSNorm(d_model), nn.Linear(d_model, ff_dim), act,
            nn.RMSNorm(ff_dim), nn.Dropout(drop), zero_module(nn.Linear(ff_dim, d_model))
            # nn.RMSNorm(d_model), act, nn.Linear(d_model, ff_dim),
            # nn.RMSNorm(ff_dim), act, nn.Linear(ff_dim, d_model)
        )

    def forward(self, x, mask=None): # [b,c,h,w], [batch, num_tok, cond_dim], [batch,T]
        bchw = x.shape
        # x = x.flatten(2).transpose(1,2) # [b,h*w,c]
        # print('attnblk fwd',x.shape)

        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.ff(x)
        # x = x.transpose(1,2).reshape(*bchw)
        # x = self.ff(x)
        # x = x + self.drop(self.ff(self.norm2(x)))
        return x


In [1]:
# @title ViT me more
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class ViT(nn.Module):
    def __init__(self, in_dim, d_model, out_dim=None, n_heads=4, nlyrs=1):
        super().__init__()
        # self.embed = nn.Linear(in_dim, d_model)
        self.embed = nn.Sequential( # in, out, kernel, stride, pad
            # nn.Conv2d(in_dim, d_model, 7, 2, 7//2, bias=False), nn.BatchNorm2d(d_model), nn.ReLU(), # ReLU SiLU
            # nn.Conv2d(d_model, d_model, 3, 2, 3//2, bias=False)

            # nn.Conv2d(in_dim, d_model//4, 7, 2, 7//2, bias=False), nn.BatchNorm2d(d_model//4), nn.ReLU(),
            # nn.Conv2d(d_model//4, d_model//2, 5, 2, 5//2, bias=False), nn.BatchNorm2d(d_model//2), nn.ReLU(),
            # nn.Conv2d(d_model//2, d_model, 3, 2, 3//2, bias=False)

            # nn.Conv2d(in_dim, d_model//4, 7, 1, 7//2, bias=False), nn.MaxPool2d(2,2), nn.BatchNorm2d(d_model//4), nn.ReLU(),
            # nn.Conv2d(d_model//4, d_model//2, 5, 1, 5//2, bias=False), nn.MaxPool2d(2,2), nn.BatchNorm2d(d_model//2), nn.ReLU(),
            # nn.Conv2d(d_model//2, d_model, 3, 1, 3//2, bias=False), nn.MaxPool2d(2,2)

            nn.Conv2d(in_dim, d_model//4, 7, 1, 7//2, bias=False), nn.BatchNorm2d(d_model//4), nn.MaxPool2d(2,2), nn.ReLU(),
            nn.Conv2d(d_model//4, d_model//2, 5, 1, 5//2, bias=False), nn.BatchNorm2d(d_model//2), nn.MaxPool2d(2,2), nn.ReLU(),
            nn.Conv2d(d_model//2, d_model, 3, 1, 3//2, bias=False), nn.MaxPool2d(2,2)

            )
        # self.pos_emb = LearnedRoPE2D(dim) # LearnedRoPE2D, RoPE2D
        # self.pos_emb = nn.Parameter(torch.randn(1,8*8,d_model)*.02)
        # self.pos_emb = nn.Parameter(GoldenGateRoPE2d((8,8), n_heads=1, d_head=d_model).affine)
        # self.pos_emb = GoldenGateRoPE2d((8,8), n_heads=1, d_head=d_model).affine
        # self.pos_emb = nn.Parameter(sincos_2d(d_model, (8,8)).unsqueeze(0))
        # self.pos_emb = sincos_2d(d_model, (8,8)).unsqueeze(0)
        self.pos_emb = sincos_2d(d_model, tuple(x//r for x in hw)).unsqueeze(0)

        self.transformer = nn.Sequential(*[AttentionBlock(d_model, n_heads) for _ in range(nlyrs)])
        # self.attn_pool = nn.Linear(d_model, 1)
        self.norm = nn.RMSNorm(d_model)
        self.out = nn.Linear(d_model, out_dim or d_model, bias=False)

    def forward(self, x, mask=None): # [b,c,h,w]
        x = self.embed(x)
        x = x.flatten(2).transpose(1,2) # [b,h*w,c]
        # print(x.shape, self.pos_emb.shape)
        x = x + self.pos_emb
        x = self.transformer(x)
        # attn = self.attn_pool(x).squeeze(-1) # [batch, (h,w)] # seq_pool
        # x = (attn.softmax(dim=1).unsqueeze(1) @ x).squeeze(1) # [batch, 1, (h,w)] @ [batch, (h,w), dim] -> [batch, dim]
        x = self.norm(x)
        return self.out(x)


# pos_emb rope < learn < learned
# conv > pixel?
# droppath not required

# norm,act,conv < conv,norm,act
# 2*s1 < uib < resblock
# gatedadaln 3 < 2 = 1 < ffmult4 = 2*gatedadaln
# MaxPool2d(2,2) < MaxPool2d(3,2,3//2)

# 32^2->8^2=64

# fla 4lyr 234449params 22.56sec
# selfattn 4lyr 213889

# ff: nla,l < nal,nal <? nla,nla < nla,nl
# dim64: head 4 < 8 = 16

# 2conv<3conv
# bn,act,conv<conv,bn,act
# silu<relu

dim = 64#64
in_dim=3
num_cls = 3
# model = ViT(image_size=32, patch_size=4, num_classes=num_classes, dim=dim, depth=1, heads=heads, mlp_dim=dim*4, channels = 3, dim_head = dim_head)
model = ViT(in_dim, 64, num_cls, nlyrs=1, n_heads=8).to(device) # 64129

print(sum(p.numel() for p in model.parameters() if p.requires_grad)) # 59850
print(sum(p.numel() for p in model.transformer[0].attn.parameters() if p.requires_grad)) # 59850
print(sum(p.numel() for p in model.transformer[0].ff.parameters() if p.requires_grad)) # 59850
optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# print(images.shape) # [batch, 3, 32, 32]
x = torch.rand(24, 3, *hw, device=device)
logits = model(x)
print(logits.shape)

# ggrope lin 50945p Accuracy: 50.6%, Avg loss: 1.364122 time: 1.6689300537109375e-06 26.679788732528685
# ggrope conv 97089p Accuracy: 67.3%, Avg loss: 0.948646 time: 1.6689300537109375e-06 17.618668341636656



NameError: name 'sincos_2d' is not defined

In [ ]:
# @title wandb
!pip install -q wandb
import wandb # https://docs.wandb.ai/quickstart
wandb.login(key='487a2109e55dce4e13fc70681781de9f50f27be7')
try: run.finish()
except NameError: pass
run = wandb.init(project="segment", config={"model": "res18",})

In [ ]:
# @title train test
import torch.nn.functional as F

def train(model, train_loader, optim):
    model.train()
    # for imgs, msks in train_loader:
    for i, (img, trg) in enumerate(train_loader): # bchw, b1hw
        img, trg = img.to(device), trg.to(device)#.flatten(1)#.long() # bchw, bt
        pred = model(img) # btd
        loss = F.cross_entropy(pred.flatten(0,1), trg.flatten(), ignore_index=-1) # [b*t,d], [b*t]
        optim.zero_grad()
        loss.backward()
        optim.step()
        try: wandb.log({"train_loss": loss.item()})
        except NameError: pass
    return loss.item()

def confusion_mat(preds, targets, num_classes, ignore_index=255): # [b,h,w]
    preds, targets = preds.flatten(), targets.flatten()
    mask = targets != ignore_index
    # mask = (target >= 0) & (target < num_classes)
    indices = targets[mask] * num_classes + preds[mask]
    confmat = torch.bincount(indices, minlength=num_classes**2).reshape(num_classes, num_classes)
    return confmat

def mIoU(confmat): # [cls,cls]
    tp = torch.diag(confmat)
    # fp, fn = confmat.sum(0) - tp, confmat.sum(1) - tp
    # iou = tp / (tp + fp + fn).clamp(min=1)
    iou = tp / (confmat.sum(1) + confmat.sum(0) - tp + 1e-6)
    return iou.mean(), iou

def test(model, test_loader):
    model.eval()
    confmat = 0
    for i, (imgs, msks) in enumerate(test_loader): # bchw, b1hw
        imgs, msks = imgs.to(device), msks.to(device)#.flatten(1)#.long() # bchw, bt
        pred = model(imgs) # btd
        loss = F.cross_entropy(pred.flatten(0,1), msks.flatten(), ignore_index=-1) # [b*t,d], [b*t]
        pred = pred.argmax(-1)
        confmat += confusion_mat(pred, msks, num_cls)
        # break
    miou, iou = mIoU(confmat)
    # print(miou, iou)
    try: wandb.log({"test_loss": loss.item(), "miou": miou.item()})
    except NameError: pass
    return miou, iou

import time
start = time.time()
for i in range(10):
    train_loss = train(model, train_loader, optim)
    test_loss = test(model, test_loader)
    print(i, time.time() - start, train_loss, test_loss)
    start = time.time()


## basement

In [ ]:

#!/bin/bash
!kaggle datasets download gopalbhattrai/pascal-voc-2012-dataset


In [ ]:
# @title chatpgt

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

root = "./data"

transform_img = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
transform_mask = transforms.Compose([transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST)])

train_dataset = datasets.VOCSegmentation(root=root, year="2012", image_set="train", download=False, transform=transform_img, target_transform=transform_mask)
val_dataset = datasets.VOCSegmentation(root=root, year="2012", image_set="val", download=False, transform=transform_img, target_transform=transform_mask)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=8)



100%|██████████| 4.52k/4.52k [00:00<00:00, 11.1MB/s]


RuntimeError: File not found or corrupted.

In [ ]:
# @title gemini
import torch
from torchvision import datasets, transforms

# 1. Define your transformations
# Note: VOC images vary in size, so resizing is usually necessary
data_transforms = transforms.Compose([transforms.Resize((448, 448)), transforms.ToTensor()])

# !mkdir -p data/
# !wget http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar -P ./data/ # 4.4kb
# !wget https://www.cs.purdue.edu/homes/skirshne/VOCtrainval_11-May-2012.tar -P ./data/ # 404
# !wget https://pjreddie.com/media/files/VOCtrainval_11-May-2012.tar -P ./data/ # 404


# https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
# 1.97gb train_val
# https://www.kaggle.com/datasets/gopalbhattrai/pascal-voc-2012-dataset/data
# 3.8gb test + train_val

# !curl -L -o ~/Downloads/pascal-voc-2012.zip https://www.kaggle.com/api/v1/datasets/download/huanghanchina/pascal-voc-2012
# !curl -o ./data/pascal-voc-2012.zip https://www.kaggle.com/api/v1/datasets/download/huanghanchina/pascal-voc-2012

import os

# Upload your kaggle.json first, then:
os.environ['KAGGLE_CONFIG_DIR'] = "/content" # or path to your json

# Use the Kaggle CLI to download (it handles the auth for you)
!pip install kaggle
!kaggle datasets download -d huanghanchina/pascal-voc-2012

# Unzip into the folder PyTorch expects
!mkdir -p ./data
!unzip -q pascal-voc-2012.zip -d ./data/


# train_seg_dataset = datasets.VOCSegmentation(root='./data', year='2012',
#     image_set='trainval', # train val trainval
#     download=False,
#     transform=data_transforms,
#     target_transform=transforms.Compose([transforms.Resize((448, 448)), transforms.ToTensor()])
# )
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [ ]:

def preprocess_mask(mask):
    mask = torch.as_tensor(np.array(mask), dtype=torch.long)
    mask[mask == 255] = -1  # ignore label
    return mask

import numpy as np

class VOCSegWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask = self.base[idx]

        # mask = torch.as_tensor(np.array(mask), dtype=torch.long)
        # mask[mask == 255] = -1
        mask = preprocess_mask(mask)

        return img, mask



In [ ]:
# @title ResBlock
import torch
import torch.nn as nn

def zero_module(module):
    for p in module.parameters():
        p.detach().zero_()
    return module

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch=None, emb_dim=None, drop=0.):
        super().__init__()
        if out_ch==None: out_ch=in_ch
        act = nn.SiLU() #
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        # self.res_conv = zero_module(nn.Conv2d(in_ch, out_ch, 1)) if in_ch != out_ch else nn.Identity()

        # self.block = nn.Sequential( # best?
        #     nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), act,
        #     zero_module(nn.Conv2d(out_ch, out_ch, 3, padding=1)), nn.BatchNorm2d(out_ch), act,
        #     )
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_ch), act, nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), act, zero_module(nn.Conv2d(out_ch, out_ch, 3, padding=1)),
            )

    def forward(self, x, emb=None): # [b,c,h,w], [batch, emb_dim]
        return self.block(x) + self.res_conv(x)


In [ ]:
# @title UpDownBlock_me
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class PixelShuffleConv(nn.Module):
    def __init__(self, in_ch, out_ch=None, kernel=3, r=1):
        super().__init__()
        self.r = r
        r = max(r, int(1/r))
        out_ch = out_ch or in_ch
        if self.r>1: self.net = nn.Sequential(nn.Conv2d(in_ch, out_ch*r**2, kernel, 1, kernel//2), nn.PixelShuffle(r)) # PixelShuffle: [b,c*r^2,h,w] -> [b,c,h*r,w*r] # upscale by upscale factor r # https://arxiv.org/pdf/1609.05158v2
        # if self.r>1: self.net = nn.Sequential(ResBlock(in_ch, out_ch*r**2, kernel), nn.PixelShuffle(r))

        elif self.r<1: self.net = nn.Sequential(nn.PixelUnshuffle(r), nn.Conv2d(in_ch*r**2, out_ch, kernel, 1, kernel//2)) # PixelUnshuffle: [b,c,h*r,w*r] -> [b,c*r^2,h,w]
        # elif self.r<1: self.net = nn.Sequential(nn.PixelUnshuffle(r), ResBlock(in_ch*r**2, out_ch, kernel))

    def forward(self, x):
        return self.net(x)

def AdaptiveAvgPool_nd(n, *args, **kwargs): return [nn.Identity, nn.AdaptiveAvgPool1d, nn.AdaptiveAvgPool2d, nn.AdaptiveAvgPool3d][n](*args, **kwargs)
def AdaptiveMaxPool_nd(n, *args, **kwargs): return [nn.Identity, nn.AdaptiveMaxPool1d, nn.AdaptiveMaxPool2d, nn.AdaptiveMaxPool3d][n](*args, **kwargs)


class AdaptivePool_at(nn.AdaptiveAvgPool1d): # AdaptiveAvgPool1d AdaptiveMaxPool1d
    def __init__(self, dim=1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.dim=dim
    def forward(self, x):
        x = x.transpose(self.dim,-1)
        shape = x.shape
        return super().forward(x.flatten(0,-2)).unflatten(0, shape[:-1]).transpose(self.dim,-1)

def make_pool_at(pool='avg', dim=1, output_size=5):
    parent={'avg':nn.AdaptiveAvgPool1d, 'max':nn.AdaptiveMaxPool1d}[pool]
    class AdaptivePool_at(parent): # AdaptiveAvgPool1d AdaptiveMaxPool1d
        def __init__(self, dim=1, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.dim=dim
        def forward(self, x):
            x = x.transpose(self.dim,-1)
            shape = x.shape
            return super().forward(x.flatten(0,-2)).unflatten(0, shape[:-1]).transpose(self.dim,-1)
    return AdaptivePool_at(dim, output_size=output_size)

class Shortcut():
    def __init__(self, dim=1, c=3, sp=(3,3), nd=2):
        self.dim = dim
        self.ch_pool = make_pool_at(pool='avg', dim=dim, output_size=c)
        # self.ch_pool = make_pool_at(pool='max', dim=dim, output_size=c)
        # self.sp_pool = AdaptiveAvgPool_nd(nd, sp)
        self.sp_pool = AdaptiveMaxPool_nd(nd, sp)

    def __call__(self, x): # [b,c,h,w]
        x = self.sp_pool(x) # spatial first preserves spatial more?
        x = self.ch_pool(x)
        return x

class UpDownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, r=1):
        super().__init__()
        act = nn.SiLU()
        self.r = r
        self.block = PixelShuffleConv(in_ch, out_ch, kernel=kernel, r=r)

        # if self.r>1: self.res_conv = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, kernel, 2, kernel//2, output_padding=1))
        # if self.r>1: self.res_conv = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'), nn.Conv2d(in_ch, out_ch, kernel, 1, kernel//2) if in_ch != out_ch else nn.Identity())
        # if self.r>1: self.res_conv = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear'), nn.Conv2d(in_ch, out_ch, kernel, 1, kernel//2) if in_ch != out_ch else nn.Identity())
        # if self.r>1: self.res_conv = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(in_ch, out_ch, kernel, 1, kernel//2) if in_ch != out_ch else nn.Identity())


        # elif self.r<1: self.res_conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel, 2, kernel//2))
        # elif self.r<1: self.res_conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel, 1, kernel//2) if in_ch != out_ch else nn.Identity(), nn.MaxPool2d(2,2))
        # elif self.r<1: self.res_conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel, 1, kernel//2) if in_ch != out_ch else nn.Identity(), nn.AvgPool2d(2,2))

        # else: self.res_conv = nn.Conv2d(in_ch, out_ch, kernel, 1, kernel//2) if in_ch != out_ch else nn.Identity()

    def forward(self, x): # bchw
        out = self.block(x)
        # # shortcut = F.interpolate(x.unsqueeze(1), size=out.shape[1:], mode='nearest-exact').squeeze(1) # pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html
        # shortcut = F.adaptive_avg_pool3d(x, out.shape[1:]) # https://pytorch.org/docs/stable/nn.html#pooling-layers
        # # shortcut = F.adaptive_max_pool3d(x, out.shape[1:]) # https://pytorch.org/docs/stable/nn.html#pooling-layers
        # # shortcut = F.adaptive_avg_pool3d(x, out.shape[1:]) if out.shape[1]>=x.shape[1] else F.adaptive_max_pool3d(x, out.shape[1:])
        # shortcut(x)
        shortcut = Shortcut(dim=1, c=out.shape[1], sp=out.shape[-2:], nd=2)(x)
        return out + shortcut
        # return out + shortcut + self.res_conv(x)
        # return out + self.res_conv(x)
        # return self.res_conv(x)

# if out>in, inter=max=ave=near.
# if out<in, inter=ave. max=max

# stride2
# interconv/convpool
# pixelconv
# pixeluib
# pixelres
# shortcut

# in_ch, out_ch = 16,3
in_ch, out_ch = 3,16
# model = UpDownBlock(in_ch, out_ch, r=1/2).to(device)
model = UpDownBlock(in_ch, out_ch, r=1/8).to(device)
# model = UpDownBlock(in_ch, out_ch, r=2).to(device)

x = torch.rand(12, in_ch, 64,64, device=device)
out = model(x)

print(out.shape)


torch.Size([12, 16, 8, 8])


In [ ]:

import torchvision.models.segmentation as models

model = models.fcn_resnet50(num_classes=21)
model = model.cuda()



In [ ]:


optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for images, masks in train_loader:
    images = images.cuda()
    masks = masks.cuda()

    outputs = model(images)["out"]  # (B, C, H, W)
    loss = F.cross_entropy(outputs, masks, ignore_index=-1)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
# @title mIoU
import torch

def confusion_mat(preds, targets, num_classes, ignore_index=255): # [b,h,w]
    preds, targets = preds.flatten(), targets.flatten()
    mask = targets != ignore_index
    # mask = (target >= 0) & (target < num_classes)
    indices = targets[mask] * num_classes + preds[mask]
    confmat = torch.bincount(indices, minlength=num_classes**2).reshape(num_classes, num_classes)
    return confmat

def mIoU(confmat): # [cls,cls]
    tp = torch.diag(confmat)
    # fp, fn = confmat.sum(0) - tp, confmat.sum(1) - tp
    # iou = tp / (tp + fp + fn).clamp(min=1)
    iou = tp / (confmat.sum(1) + confmat.sum(0) - tp + 1e-6)
    return iou.mean(), iou


            # preds = torch.argmax(preds, dim=1)
            preds = preds.argmax(-1)
            confmat += confusion_mat(preds, targets, num_classes)
        miou, per_class_iou = compute_miou(confmat)


In [ ]:
# https://www.kaggle.com/code/ligtfeather/semantic-segmentation-is-easy-with-pytorch

def mIoU(pred_mask, mask, smooth=1e-10, n_classes=23):
    with torch.no_grad():
        pred_mask = F.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1)
        pred_mask = pred_mask.contiguous().view(-1)
        mask = mask.contiguous().view(-1)

        iou_per_class = []
        for clas in range(0, n_classes): #loop per pixel class
            true_class = pred_mask == clas
            true_label = mask == clas

            if true_label.long().sum().item() == 0: #no exist label in this loop
                iou_per_class.append(np.nan)
            else:
                intersect = torch.logical_and(true_class, true_label).sum().float().item()
                union = torch.logical_or(true_class, true_label).sum().float().item()

                iou = (intersect + smooth) / (union +smooth)
                iou_per_class.append(iou)
        return np.nanmean(iou_per_class)


In [ ]:
# @title iouCalc
# https://github.com/hoya012/semantic-segmentation-tutorial-pytorch/blob/master/helpers/helpers.py
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

"""
====================
Classes for logging and progress printing.
====================
"""
# class AverageMeter(object):
#     """Computes and stores the average and current value"""
#     def __init__(self, name, fmt=':f'):
#         self.name = name
#         self.fmt = fmt
#         self.reset()

#     def reset(self):
#         self.val = 0
#         self.avg = 0
#         self.sum = 0
#         self.count = 0

#     def update(self, val, n=1):
#         self.val = val
#         self.sum += val * n
#         self.count += n
#         self.avg = self.sum / self.count

#     def __str__(self):
#         fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
#         return fmtstr.format(**self.__dict__)


# class ProgressMeter(object):
#     def __init__(self, num_batches, meters, prefix=""):
#         self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
#         self.meters = meters
#         self.prefix = prefix

#     def display(self, batch):
#         entries = [self.prefix + self.batch_fmtstr.format(batch)]
#         entries += [str(meter) for meter in self.meters]
#         print('\t'.join(entries))

#     def _get_batch_fmtstr(self, num_batches):
#         num_digits = len(str(num_batches // 1))
#         fmt = '{:' + str(num_digits) + 'd}'
#         return '[' + fmt + '/' + fmt.format(num_batches) + ']'

"""
====================
iouCalc: Calculates IoU scores of dataset per individual batch.
Arguments:
    - labelNames: list of class names
    - validClasses: list of valid class ids
    - voidClass: class id of void class (class ignored for calculating metrics)
====================
"""

class iouCalc():
    def __init__(self, classLabels, validClasses, voidClass = None):
        assert len(classLabels) == len(validClasses), 'Number of class ids and names must be equal'
        self.classLabels    = classLabels
        self.validClasses   = validClasses
        self.voidClass      = voidClass
        self.evalClasses    = [l for l in validClasses if l != voidClass]

        self.perImageStats  = []
        self.nbPixels       = 0
        self.confMatrix     = np.zeros(shape=(len(self.validClasses),len(self.validClasses)),dtype=np.ulonglong)

        # Init IoU log files
        self.headerStr = 'epoch, '
        for label in self.classLabels:
            if label.lower() != 'void':
                self.headerStr += label + ', '

    def clear(self):
        self.perImageStats  = []
        self.nbPixels       = 0
        self.confMatrix     = np.zeros(shape=(len(self.validClasses),len(self.validClasses)),dtype=np.ulonglong)

    def getIouScoreForLabel(self, label):
        # Calculate and return IOU score for a particular label (train_id)
        if label == self.voidClass:
            return float('nan')

        # the number of true positive pixels for this label
        # the entry on the diagonal of the confusion matrix
        tp = np.longlong(self.confMatrix[label,label])

        # the number of false negative pixels for this label
        # the row sum of the matching row in the confusion matrix
        # minus the diagonal entry
        fn = np.longlong(self.confMatrix[label,:].sum()) - tp

        # the number of false positive pixels for this labels
        # Only pixels that are not on a pixel with ground truth label that is ignored
        # The column sum of the corresponding column in the confusion matrix
        # without the ignored rows and without the actual label of interest
        notIgnored = [l for l in self.validClasses if not l == self.voidClass and not l==label]
        fp = np.longlong(self.confMatrix[notIgnored,label].sum())

        # the denominator of the IOU score
        denom = (tp + fp + fn)
        if denom == 0:
            return float('nan')

        # return IOU
        return float(tp) / denom

    def evaluateBatch(self, predictionBatch, groundTruthBatch):
        # Calculate IoU scores for single batch
        assert predictionBatch.size(0) == groundTruthBatch.size(0), 'Number of predictions and labels in batch disagree.'

        # Load batch to CPU and convert to numpy arrays
        predictionBatch = predictionBatch.cpu().numpy()
        groundTruthBatch = groundTruthBatch.cpu().numpy()

        for i in range(predictionBatch.shape[0]):
            predictionImg = predictionBatch[i,:,:]
            groundTruthImg = groundTruthBatch[i,:,:]

            # Check for equal image sizes
            assert predictionImg.shape == groundTruthImg.shape, 'Image shapes do not match.'
            assert len(predictionImg.shape) == 2, 'Predicted image has multiple channels.'

            imgWidth  = predictionImg.shape[0]
            imgHeight = predictionImg.shape[1]
            nbPixels  = imgWidth*imgHeight

            # Evaluate images
            encoding_value = max(groundTruthImg.max(), predictionImg.max()).astype(np.int32) + 1
            encoded = (groundTruthImg.astype(np.int32) * encoding_value) + predictionImg

            values, cnt = np.unique(encoded, return_counts=True)

            for value, c in zip(values, cnt):
                pred_id = value % encoding_value
                gt_id = int((value - pred_id)/encoding_value)
                if not gt_id in self.validClasses:
                    printError('Unknown label with id {:}'.format(gt_id))
                self.confMatrix[gt_id][pred_id] += c

            # Calculate pixel accuracy
            notIgnoredPixels = np.in1d(groundTruthImg, self.evalClasses, invert=True).reshape(groundTruthImg.shape)
            erroneousPixels = np.logical_and(notIgnoredPixels, (predictionImg != groundTruthImg))
            nbNotIgnoredPixels = np.count_nonzero(notIgnoredPixels)
            nbErroneousPixels = np.count_nonzero(erroneousPixels)
            self.perImageStats.append([nbNotIgnoredPixels, nbErroneousPixels])

            self.nbPixels += nbPixels

        return

    def outputScores(self):
        # Output scores over dataset
        assert self.confMatrix.sum() == self.nbPixels, 'Number of analyzed pixels and entries in confusion matrix disagree: confMatrix {}, pixels {}'.format(self.confMatrix.sum(),self.nbPixels)

        # Calculate IOU scores on class level from matrix
        classScoreList = []

        # Print class IOU scores
        outStr = 'classes           IoU\n'
        outStr += '---------------------\n'
        for c in self.evalClasses:
            iouScore = self.getIouScoreForLabel(c)
            classScoreList.append(iouScore)
            outStr += '{:<14}: {:>5.3f}\n'.format(self.classLabels[c], iouScore)
        miou = getScoreAverage(classScoreList)
        outStr += '---------------------\n'
        outStr += 'Mean IoU      : {avg:5.3f}\n'.format(avg=miou)
        outStr += '---------------------'
        print(outStr)
        return miou

# # Print an error message and quit
# def printError(message):
#     print('ERROR: ' + str(message))
#     sys.exit(-1)

def getScoreAverage(scoreList):
    validScores = 0
    scoreSum    = 0.0
    for score in scoreList:
        if not np.isnan(score):
            validScores += 1
            scoreSum += score
    if validScores == 0:
        return float('nan')
    return scoreSum / validScores


"""
================
Visualize images
================
"""

# def visim(img, args):
#     img = img.cpu()
#     # Convert image data to visual representation
#     img *= torch.tensor(args.dataset_std)[:,None,None]
#     img += torch.tensor(args.dataset_mean)[:,None,None]
#     npimg = (img.numpy()*255).astype('uint8')
#     if len(npimg.shape) == 3 and npimg.shape[0] == 3:
#         npimg = np.transpose(npimg, (1, 2, 0))
#     else:
#         npimg = npimg[0,:,:]
#     return npimg

# def vislbl(label, mask_colors):
#     label = label.cpu()
#     # Convert label data to visual representation
#     label = np.array(label.numpy())
#     if label.shape[-1] == 1:
#         label = label[:,:,0]

#     # Convert train_ids to colors
#     label = mask_colors[label]
#     return label

# """
# ====================
# Plot learning curves
# ====================
# """

# def plot_learning_curves(metrics, args):
#     x = np.arange(args.epochs)
#     fig, ax1 = plt.subplots()
#     ax1.set_xlabel('epochs')
#     ax1.set_ylabel('loss')
#     ln1 = ax1.plot(x, metrics['train_loss'], color='tab:red')
#     ln2 = ax1.plot(x, metrics['val_loss'], color='tab:red', linestyle='dashed')
#     ax1.grid()
#     ax2 = ax1.twinx()
#     ax2.set_ylabel('accuracy')
#     ln3 = ax2.plot(x, metrics['train_acc'], color='tab:blue')
#     ln4 = ax2.plot(x, metrics['val_acc'], color='tab:blue', linestyle='dashed')
#     ln5 = ax2.plot(x, metrics['miou'], color='tab:green')
#     lns = ln1+ln2+ln3+ln4+ln5
#     plt.legend(lns, ['Train loss','Validation loss','Train accuracy','Validation accuracy','mIoU'])
#     plt.tight_layout()
#     plt.savefig(args.save_path + '/learning_curve.png', bbox_inches='tight')




In [ ]:
# @title train val func
# https://github.com/hoya012/semantic-segmentation-tutorial-pytorch/blob/master/learning/learner.py#L18
from helpers.helpers import AverageMeter, ProgressMeter, iouCalc, visim, vislbl
from learning.minicity import MiniCity
from learning.utils import rand_bbox, copyblob
import torch
import torch.nn.functional as F
import cv2
import os
import numpy as np
import time
from PIL import Image

"""
=================
Routine functions
=================
"""

def train_epoch(dataloader, model, criterion, optimizer, lr_scheduler, epoch, void=-1, args=None):
    batch_time = AverageMeter('Time', ':6.3f')
    data_time = AverageMeter('Data', ':6.3f')
    loss_running = AverageMeter('Loss', ':.4e')
    acc_running = AverageMeter('Accuracy', ':.3f')
    progress = ProgressMeter(
        len(dataloader),
        [batch_time, data_time, loss_running, acc_running],
        prefix="Train, epoch: [{}]".format(epoch))

    # input resolution
    if args.crop_size is not None:
        res = args.crop_size[0]*args.crop_size[1]
    else:
        res = args.train_size[0]*args.train_size[1]

    # Set model in training mode
    model.train()

    end = time.time()

    with torch.set_grad_enabled(True):
        # Iterate over data.
        for epoch_step, (inputs, labels, _) in enumerate(dataloader):
            data_time.update(time.time()-end)

            if args.copyblob:
                for i in range(inputs.size()[0]):
                    rand_idx = np.random.randint(inputs.size()[0])
                    # # wall(3) --> sidewalk(1)
                    # copyblob(src_img=inputs[i], src_mask=labels[i], dst_img=inputs[rand_idx], dst_mask=labels[rand_idx], src_class=3, dst_class=1)
                    # # fence(4) --> sidewalk(1)
                    # copyblob(src_img=inputs[i], src_mask=labels[i], dst_img=inputs[rand_idx], dst_mask=labels[rand_idx], src_class=4, dst_class=1)
                    # # bus(15) --> road(0)
                    # copyblob(src_img=inputs[i], src_mask=labels[i], dst_img=inputs[rand_idx], dst_mask=labels[rand_idx], src_class=15, dst_class=0)
                    # # train(16) --> road(0)
                    # copyblob(src_img=inputs[i], src_mask=labels[i], dst_img=inputs[rand_idx], dst_mask=labels[rand_idx], src_class=16, dst_class=0)

            inputs = inputs.float().cuda()
            labels = labels.long().cuda()

            # zero the parameter gradients
            optimizer.zero_grad()

            # if args.cutmix:
            #     # generate mixed sample
            #     lam = np.random.beta(1., 1.)
            #     rand_index = torch.randperm(inputs.size()[0]).cuda()
            #     bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
            #     inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
            #     labels[:, bbx1:bbx2, bby1:bby2] = labels[rand_index, bbx1:bbx2, bby1:bby2]

            # forward pass
            outputs = model(inputs)
            outputs = outputs['out'] #FIXME for DeepLab V3
            preds = torch.argmax(outputs, 1)
            # cross-entropy loss
            loss = criterion(outputs, labels)

            # backward pass
            loss.backward()
            optimizer.step()

            # Statistics
            bs = inputs.size(0) # current batch size
            loss = loss.item()
            loss_running.update(loss, bs)
            corrects = torch.sum(preds == labels.data)
            nvoid = int((labels==void).sum())
            acc = corrects.double()/(bs*res-nvoid) # correct/(batch_size*resolution-voids)
            acc_running.update(acc, bs)

            # output training info
            progress.display(epoch_step)

            # Measure time
            batch_time.update(time.time() - end)
            end = time.time()

        # Reduce learning rate
        lr_scheduler.step(loss_running.avg)

    return loss_running.avg, acc_running.avg


def validate_epoch(dataloader, model, criterion, epoch, classLabels, validClasses, void=-1, maskColors=None, folder='baseline_run', args=None):
    batch_time = AverageMeter('Time', ':6.3f')
    data_time = AverageMeter('Data', ':6.3f')
    loss_running = AverageMeter('Loss', ':.4e')
    acc_running = AverageMeter('Accuracy', ':.4e')
    iou = iouCalc(classLabels, validClasses, voidClass = void)
    progress = ProgressMeter(
        len(dataloader),
        [batch_time, data_time, loss_running, acc_running],
        prefix="Test, epoch: [{}]".format(epoch))

    # input resolution
    res = args.test_size[0]*args.test_size[1]

    # Set model in evaluation mode
    model.eval()

    with torch.no_grad():
        end = time.time()
        for epoch_step, (inputs, labels, filepath) in enumerate(dataloader):
            data_time.update(time.time()-end)

            inputs = inputs.float().cuda()
            labels = labels.long().cuda()

            # forward
            outputs = model(inputs)
            outputs = outputs['out'] #FIXME
            preds = torch.argmax(outputs, 1)
            loss = criterion(outputs, labels)

            # Statistics
            bs = inputs.size(0) # current batch size
            loss = loss.item()
            loss_running.update(loss, bs)
            corrects = torch.sum(preds == labels.data)
            nvoid = int((labels==void).sum())
            acc = corrects.double()/(bs*res-nvoid) # correct/(batch_size*resolution-voids)
            acc_running.update(acc, bs)
            # Calculate IoU scores of current batch
            iou.evaluateBatch(preds, labels)

            # Save visualizations of first batch
            if epoch_step == 0 and maskColors is not None:
                for i in range(inputs.size(0)):
                    filename = os.path.splitext(os.path.basename(filepath[i]))[0]
                    # Only save inputs and labels once
                    if epoch == 0:
                        img = visim(inputs[i,:,:,:], args)
                        label = vislbl(labels[i,:,:], maskColors)
                        if len(img.shape) == 3:
                            cv2.imwrite(folder + '/images/{}.png'.format(filename),img[:,:,::-1])
                        else:
                            cv2.imwrite(folder + '/images/{}.png'.format(filename),img)
                        cv2.imwrite(folder + '/images/{}_gt.png'.format(filename),label[:,:,::-1])
                    # Save predictions
                    pred = vislbl(preds[i,:,:], maskColors)
                    cv2.imwrite(folder + '/images/{}_epoch_{}.png'.format(filename,epoch),pred[:,:,::-1])

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            # print progress info
            progress.display(epoch_step)

        miou = iou.outputScores()
        print('Accuracy      : {:5.3f}'.format(acc_running.avg))
        print('---------------------')

    return acc_running.avg, loss_running.avg, miou

def predict(dataloader, model, maskColors, folder='baseline_run', mode='val', args=None):
    batch_time = AverageMeter('Time', ':6.3f')
    data_time = AverageMeter('Data', ':6.3f')
    progress = ProgressMeter(
        len(dataloader),
        [batch_time, data_time],
        prefix='Predict: ')

    Dataset = MiniCity

    # Set model in evaluation mode
    model.eval()

    with torch.no_grad():
        end = time.time()
        for epoch_step, batch in enumerate(dataloader):

            if len(batch) == 2:
                inputs, filepath = batch
            else:
                inputs, _, filepath = batch

            data_time.update(time.time()-end)

            inputs = inputs.float().cuda()

            if args.mst:
                batch_idx, _, h, w = inputs.size() #(1, 20, 1024, 2048)
                # only single image is supported for multi-scale testing
                assert(batch_idx == 1)
                with torch.cuda.device_of(inputs):
                    scores = inputs.new().resize_(batch_idx, len(Dataset.validClasses), h, w).zero_().cuda()

                scales = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.2] #FIXME

                for scale in scales:
                    inputs_resized = F.interpolate(inputs, scale_factor=scale, mode='bilinear', align_corners=True)
                    #print("original size {}x{} --> resized to {}x{}".format(h, w, inputs_resized.size()[2], inputs_resized.size()[3]))

                    # forward
                    outputs = model(inputs_resized)
                    outputs = outputs['out'] #FIXME (1, 20, 512, 1024) for scale 0.5

                    score = F.interpolate(outputs, (h, w), mode='bilinear', align_corners=True)
                    scores += score


                    # forward using flipped input
                    with torch.cuda.device_of(inputs_resized):
                        idx = torch.arange(inputs_resized.size(3)-1, -1, -1).type_as(inputs_resized).long()
                    input_resized_flip = inputs_resized.index_select(3, idx)

                    # forward
                    outputs = model(input_resized_flip)
                    outputs = outputs['out'] #FIXME
                    outputs = outputs.index_select(3, idx)

                    score = F.interpolate(outputs, (h, w), mode='bilinear', align_corners=True)
                    scores += score

                # averaging scores
                scores = scores / (2*len(scales))

                preds = torch.argmax(scores, 1) # (1, 512, 1024)
            else:
                # forward
                outputs = model(inputs)
                outputs = outputs['out'] #FIXME

                preds = torch.argmax(outputs, 1)

            # Save visualizations of first batch
            for i in range(inputs.size(0)):
                filename = os.path.splitext(os.path.basename(filepath[i]))[0]
                # Save input
                img = visim(inputs[i,:,:,:], args)
                img = Image.fromarray(img, 'RGB')
                img.save(folder + '/results_color_{}/{}_input.png'.format(mode, filename))
                # Save prediction with color labels
                pred = preds[i,:,:].cpu()
                pred_color = vislbl(pred, maskColors)
                pred_color = Image.fromarray(pred_color.astype('uint8'))
                pred_color.save(folder + '/results_color_{}/{}_prediction.png'.format(mode, filename))
                # Save class id prediction (used for evaluation)
                pred_id = Dataset.trainid2id[pred]
                pred_id = Image.fromarray(pred_id)
                pred_id = pred_id.resize((2048,1024), resample=Image.NEAREST)
                pred_id.save(folder + '/results_{}/{}.png'.format(mode, filename))


            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            # print progress info
            progress.display(epoch_step)
